# Tiny Aya Global + the Hinglish PlainSFT adapter: generate and score (Colab, T4)

Loads the base model and the published LoRA adapter at pinned commits, generates greedy replies, and scores each with this repo's GlotLID language-kept scorer.

- **The 16 over-refusal prompts** (harmless questions that sound risky; never used in training): base vs adapter, replies shown.
- **The 42 training prompts**: adapter only. These were used in training, so this shows what the adapter learned, not how it generalises. For the 18 refusal rows (harmful requests) only the scores are printed, never the reply text.

**Before you run:**
1. Runtime > Change runtime type > **T4 GPU**. The notebook stops on CPU.
2. Open [CohereLabs/tiny-aya-global](https://huggingface.co/CohereLabs/tiny-aya-global) logged in and accept Cohere's terms (approval is automatic).
3. Create a Hugging Face **read** token and add it in Colab's Secrets panel (key icon) as `HF_TOKEN`, with notebook access on.

About 15 minutes on a T4, of which ~7 GB of downloads.

In [ ]:
# Pinned packages (the versions that loaded base + adapter in the 2026-11-09 reproduction). torch: Colab's own.
!pip install -q transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 fasttext==0.9.3
!git clone -q https://github.com/kazani-351/tiny-aya-hinglish-safety
import os, sys
os.chdir("tiny-aya-hinglish-safety"); sys.path.insert(0, os.getcwd())

In [ ]:
import csv, time, torch, transformers, peft
from huggingface_hub import hf_hub_download, login
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run all again."
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    raise SystemExit(f"HF login failed ({type(e).__name__}): add HF_TOKEN in Colab Secrets and allow notebook access.")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "|", torch.cuda.get_device_name(0))
BASE = ("CohereLabs/tiny-aya-global", "af89d219b53ed9b13b8a4645f9c8028973510324")
ADAPTER = ("kazani351/tiny-aya-global-hinglish-lora", "cc880b1725f5beec0f4a8657f51b52eb3e10bd5a")
DATA = ("kazani351/tiny-aya-hinglish-safety-data", "ca83fcea77aa4a70e674c12425f56b05d28c64cb")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
t = time.perf_counter()
tok = AutoTokenizer.from_pretrained(BASE[0], revision=BASE[1])
base = AutoModelForCausalLM.from_pretrained(BASE[0], revision=BASE[1], dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(base, ADAPTER[0], revision=ADAPTER[1]).eval()
print(f"loaded in {time.perf_counter()-t:.0f}s")

def generate(prompt, use_adapter=True):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to("cuda")
    with torch.no_grad():
        if use_adapter:
            out = model.generate(**enc, max_new_tokens=160, do_sample=False)
        else:
            with model.disable_adapter():
                out = model.generate(**enc, max_new_tokens=160, do_sample=False)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

import scorers
predict = scorers.glotlid_predict()
def score(reply):
    return scorers.lang_kept(predict, "hinglish", reply), "_".join(scorers.top_label(predict, reply))
def rows(name):
    return list(csv.DictReader(open(hf_hub_download(DATA[0], name, repo_type="dataset", revision=DATA[1]), encoding="utf-8")))

## 1. Over-refusal prompts (held out): base vs adapter

In [ ]:
pseudo = rows("pseudo_harmful_hinglish.csv")
kept = {"base": 0, "adapter": 0}
for r in pseudo:
    print(f"--- {r['id']}: {r['prompt']}")
    for name, flag in (("base", False), ("adapter", True)):
        reply = generate(r["prompt"], use_adapter=flag)
        k, lab = score(reply); kept[name] += k
        print(f"[{name} | Hinglish kept={k:.0f} | {lab}] {reply}\n")
print(f"Hinglish kept, {len(pseudo)} over-refusal prompts: base {kept['base']:.0f}, adapter {kept['adapter']:.0f}")

## 2. The 42 training prompts: adapter (reply text hidden for the 18 harmful requests)

In [ ]:
train = rows("train_hinglish.csv")
assert len(train) == 42
total = 0
for r in train:
    reply = generate(r["prompt"])
    k, lab = score(reply); total += k
    shown = "(reply hidden: harmful request)" if r["type"] == "refuse" else reply[:200]
    print(f"{r['id']} {r['type']:<20} kept={k:.0f} {lab:<9} {shown}")
print(f"Hinglish kept on the 42 training prompts: {total:.0f}/42")